<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20261/Module%206%20-%20Introduction%20to%20CrewAI/CrewAI_Basics_lab.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px;">
    </a>
</div>

<div style="background-color:#F5EDF4; border-top:3px solid #A83296; padding:16px; border-radius:6px">
<h1 style="color:#493347; margin:0">CrewAI Basics: the Building Blocks, One at a Time</h1>
<p style="color:#56616D; margin:6px 0 0 0">307307 BI Methods for Generative AI - University of Petra</p>
</div>

This is the **gentle introduction** to CrewAI. Start here, then move on to the longer lab (*Building AI Teams with CrewAI*), which adds data, knowledge, memory and guardrails.

Each section adds **one** idea to the one before it, and every idea comes with a small example you can run and change.

| Part | Building block | In one sentence |
|---|---|---|
| 1 | **Agent** | An LLM with a job description (role, goal, backstory) |
| 2 | **Task** and **Crew** | A piece of work with a clear "done", and the container that runs it |
| 3 | **Several agents** | Agents passing work along, like an assembly line |
| 4 | **Tool** | A Python function the agent may call to get facts or act |
| 5 | **Structured output** | An answer your code can use, not just a paragraph |
| 6 | **Flow** | You decide the order of the steps (`@start`, `@listen`, `@router`) |
| 7 | **Flow + Crew** | A Flow for the skeleton, a Crew for the thinking |

Parts 1 to 5 need a Gemini API key. Part 6 uses **no AI at all**, so you can see exactly how a Flow works.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Setup</h2>
</div>

Install CrewAI with Gemini support and give it your key. In Google Colab store the key as `GOOGLE_API_KEY` in the Secrets panel (key icon on the left); elsewhere the cell asks you to paste it. The first install takes one to two minutes; if Colab asks you to restart the session, do so and run this cell again.

In [ ]:
# Lines starting with "!" are shell commands. Remove the "!" if you run this in a terminal.
!pip install -q "crewai[google-genai]"

In [ ]:
import os, getpass

# Read the API key from Colab Secrets if available, otherwise ask for it.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

# Turn off anonymous usage reporting.
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Task, Crew, Process, LLM

# The prefix "gemini/" selects Google Gemini. If you get "model not found", change only this line.
llm = LLM(model="gemini/gemini-3.6-flash")
print("Ready")

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 1: The Agent</h2>
</div>

An **agent** is a language model with a *job description*:

- **role**: who it is ("Travel Planner")
- **goal**: what it tries to achieve
- **backstory**: how it behaves and what it values

CrewAI turns these three sentences into the instructions the model receives. Write them as you would write a job advert.

`agent.kickoff("...")` runs the agent once on a question. Nothing else is needed.

In [ ]:
guide = Agent(
    role="Student Travel Guide",
    goal="Give short, practical travel advice to students on a small budget",
    backstory="You have backpacked across 40 countries and you always think about cost first.",
    llm=llm,
)

answer = guide.kickoff("I have 2 days in Amman. What should I see?")
print(answer.raw)

**Try it.** Change only the `backstory` (for example: *"You only recommend luxury hotels"*) and run the cell again. Same model, same question, very different answer. The persona matters.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 2: Task and Crew</h2>
</div>

A chat question is vague. A **Task** is a precise piece of work with two parts:

- `description`: what to do
- `expected_output`: what a finished result looks like (this is your most powerful control)

A **Crew** is the container that holds agents and tasks and runs them with `kickoff()`.

| Component | Answers |
|---|---|
| Agent | *Who* does the work? |
| Task | *What* exactly, and what is "done"? |
| Crew | *Which* agents and tasks run together? |

In [ ]:
explain = Task(
    description="Explain what an API is to a 12-year-old.",
    expected_output="Exactly 3 sentences, including one everyday analogy.",
    agent=guide,
)

crew = Crew(agents=[guide], tasks=[explain])
result = crew.kickoff()
print(result.raw)

**Try it.** Change `expected_output` to *"A single sentence of at most 15 words"* and run again. Notice how much control one line gives you.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 3: Several agents passing work along</h2>
</div>

Real work often needs different skills. Here a **researcher** collects facts and a **writer** turns them into a short summary. By default a crew runs tasks **one after another** (`Process.sequential`), like an assembly line, and each task receives the earlier results.

Two small features make the crew reusable:

- `{topic}` is a **placeholder** in a task description. Its value is supplied at `kickoff(inputs={...})`.
- `context=[research]` says explicitly: *this task needs the output of that task*.

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="List accurate, relevant facts",
    backstory="Thorough and sceptical. You never invent facts.",
    llm=llm,
)
writer = Agent(
    role="Writer",
    goal="Write crisp summaries for busy managers",
    backstory="You write short, clear business prose.",
    llm=llm,
)

research = Task(
    description="List 5 facts about {topic}.",
    expected_output="A bulleted list of 5 facts.",
    agent=researcher,
)
write = Task(
    description="Turn the facts into a summary for a manager.",
    expected_output="One paragraph of at most 50 words.",
    agent=writer,
    context=[research],        # the writer receives the researcher's output
)

team = Crew(agents=[researcher, writer], tasks=[research, write], process=Process.sequential)

print(team.kickoff(inputs={"topic": "solar power in Jordan"}).raw)

**Try it.** Run the *same* `team` again with `inputs={"topic": "electric cars"}`. One crew, any topic. Then make `research.expected_output` vague (*"Some facts"*) and see how the quality drops.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 4: Tools</h2>
</div>

A language model is fluent but **bad at exact arithmetic and live facts**. A **tool** is an ordinary Python function the agent may choose to call.

The agent never sees your code. It sees the tool's **name**, its **type hints** and its **docstring** (the text in triple quotes). That text is how it decides *when* and *how* to use the tool, so write it carefully.

Remember *intent vs action* from Module 4: the model only *asks* for the tool; CrewAI runs it and gives the result back.

In [ ]:
from crewai.tools import tool

@tool("Add VAT")
def add_vat(price: float, vat_percent: float) -> str:
    """Calculate the final price after adding VAT (sales tax).
    Inputs: the price before tax, and the VAT rate as a percentage (for example 16 for 16%).
    Returns the price including VAT, rounded to 2 decimals."""
    return str(round(price * (1 + vat_percent / 100), 2))

# You can test a tool directly, with no AI involved:
print(add_vat.run(price=80, vat_percent=16))

Now give the tool to an agent. The only change is `tools=[...]`. Watch the printed log: you will see the agent decide to call `add_vat`, the numbers it sent, and the result it received.

In [ ]:
shopper = Agent(
    role="Shopping Assistant",
    goal="Answer price questions exactly, never guessing a number you can calculate",
    backstory="You help customers understand what they will really pay.",
    tools=[add_vat],
    llm=llm,
    verbose=True,
)

price_task = Task(
    description="A laptop costs 849.50 before tax. VAT is 16%. What is the final price?",
    expected_output="One sentence stating the final price.",
    agent=shopper,
)

print(Crew(agents=[shopper], tasks=[price_task]).kickoff().raw)

**Try it.** Make the docstring vague (delete the "Inputs:" lines) and ask again. Does the agent still use the tool correctly? A weak docstring is the most common reason an agent misuses a tool.

**Exercise.** Write a tool `tip_calculator(bill, tip_percent)` that returns the tip amount, test it with `.run(...)`, and give it to an agent.

In [ ]:
# YOUR CODE HERE



# ---- Solution (remove the # signs to try it) ----
# @tool("Tip calculator")
# def tip_calculator(bill: float, tip_percent: float) -> str:
#     """Calculate the tip for a restaurant bill.
#     Inputs: the bill amount and the tip as a percentage (for example 10 for 10%).
#     Returns the tip amount rounded to 2 decimals."""
#     return str(round(bill * tip_percent / 100, 2))
#
# print(tip_calculator.run(bill=45, tip_percent=10))   # expect 4.5

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 5: Structured output</h2>
</div>

Free text is hard to use in a program. If a later step must read a *field* (a score, a yes/no, a list), ask for a typed object. We describe its shape with a small Python class (a **Pydantic model**) and give it to the task as `output_pydantic`.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel

class TripVerdict(BaseModel):
    worth_it: Literal["yes", "maybe", "no"]    # only these three values are allowed
    reasons: List[str]

verdict_task = Task(
    description="A student asks: is a 1-day trip from Amman to Petra worth the cost?",
    expected_output="A verdict (yes, maybe or no) and 2 to 3 short reasons.",
    agent=guide,
    output_pydantic=TripVerdict,
)

out = Crew(agents=[guide], tasks=[verdict_task]).kickoff()

print(out.pydantic.worth_it)       # a real Python value, usable in an if-statement
print(out.pydantic.reasons)

**Check.** `out.pydantic.worth_it` is always one of three exact words. That is what lets the next part make a decision with a plain `if`.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 6: Flows (no AI needed)</h2>
</div>

A crew lets agents *decide*. Sometimes you want **you** to decide the order: do this, then that, and *if* something then do something else. That is a **Flow**.

A Flow is a normal Python class where **each method is a step**. Decorators say *when* a step runs:

| Decorator | Meaning |
|---|---|
| `@start()` | Run this when the flow begins (the entry point) |
| `@listen(step)` | Run this after `step` finishes; it receives `step`'s return value |
| `@router(step)` | Run after `step`, then **choose a path** by returning a text label |
| `@listen("label")` | Run when a router returns that label |

Think of **dominoes**: `@start` tips the first one and `@listen` says which falls next. `self.state` is memory shared by every step.

First a Flow with **no AI at all**, so you only see the mechanics. This one needs no API key.

In [ ]:
from crewai.flow.flow import Flow, start, listen, router
from pydantic import BaseModel

class OrderState(BaseModel):
    amount: float = 0

class OrderFlow(Flow[OrderState]):

    @start()
    def receive_order(self):
        print("1. order received")
        self.state.amount = 250          # shared memory for the whole flow
        return "order-17"

    @listen(receive_order)
    def log_order(self, order_id):       # gets receive_order's return value
        print("2. logging", order_id, "amount =", self.state.amount)

    @router(receive_order)
    def check_amount(self):
        return "needs_approval" if self.state.amount > 100 else "auto_approve"

    @listen("needs_approval")
    def ask_manager(self):
        print("3. amount is large, a manager must approve")

    @listen("auto_approve")
    def approve(self):
        print("3. small amount, approved automatically")

OrderFlow().kickoff()

Read the output and match it to the code: `receive_order` ran first (`@start`); `log_order` and the router both ran after it; the router returned `"needs_approval"`, so **only** `ask_manager` ran, never `approve`. That is an if/else.

**Questions.**
1. Change `self.state.amount = 250` to `50` and predict the output before you run it. Does `log_order` still run?
2. Why is the router better than an `if` buried inside one big function? (Hint: each step is small, named and easy to read.)

In [ ]:
# YOUR CODE HERE: change the amount, run OrderFlow().kickoff() again

A flow can also **wait for several steps**. With `and_(a, b)` a step runs only after *both* finish; with `or_(a, b)` as soon as *either* finishes.

In [ ]:
from crewai.flow.flow import and_

class CheckFlow(Flow):
    @start()
    def check_stock(self):
        print("stock checked")

    @start()
    def check_payment(self):
        print("payment checked")

    @listen(and_(check_stock, check_payment))
    def ship(self):
        print("both checks done: shipping")

CheckFlow().kickoff()

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 7: Flow + Crew together</h2>
</div>

The usual real-world pattern: **a Flow for the skeleton, Crews for the thinking.** Each step of a Flow can be plain Python, one agent, or a whole crew.

Here we do a cheap one-agent check first (Part 5's verdict). A router then **stops early** on a "no" and only spends on the full two-agent team (Part 3) when the answer is promising. This saves money and makes the process predictable.

In [ ]:
class TripState(BaseModel):
    question: str = ""
    worth_it: str = ""
    plan: str = ""

class TripFlow(Flow[TripState]):

    @start()
    def quick_check(self):
        t = Task(
            description="Quick opinion for a student: {question}",
            expected_output="A verdict (yes, maybe or no) and 2 to 3 short reasons.",
            agent=guide,
            output_pydantic=TripVerdict,
        )
        out = Crew(agents=[guide], tasks=[t]).kickoff(inputs={"question": self.state.question})
        self.state.worth_it = out.pydantic.worth_it

    @router(quick_check)
    def decide(self):
        return "stop" if self.state.worth_it == "no" else "go"

    @listen("stop")
    def say_no(self):
        print("Quick check says no. Stopping here, the full team was never used.")

    @listen("go")
    def full_plan(self):
        r = Task(description="List 5 facts a student should know before this trip: {question}",
                 expected_output="A bulleted list of 5 facts.", agent=researcher)
        w = Task(description="Write a short plan from the facts.",
                 expected_output="One paragraph of at most 60 words.", agent=writer, context=[r])
        out = Crew(agents=[researcher, writer], tasks=[r, w]).kickoff(inputs={"question": self.state.question})
        self.state.plan = out.raw
        print(self.state.plan)

flow = TripFlow()
flow.state.question = "a weekend trip from Amman to Aqaba"
flow.kickoff()

**Try it.** Run the flow with a question that should get a "no" (for example *"a 1-day trip from Amman to Tokyo"*) and watch it stop early.

## Crew or Flow?

| | **Crew** | **Flow** |
|---|---|---|
| Who decides the order? | The agents | You, in code |
| Strength | Open-ended, flexible thinking | Predictable, cheap, easy to audit |
| Use for | "Research this and write a summary" | "Check first; only then spend on the full team" |

In practice you combine them, exactly as in Part 7.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Summary</h2>
</div>

- An **Agent** is a model with a role, goal and backstory. A **Task** says what to do and what "done" looks like (`expected_output`). A **Crew** runs them.
- With several agents, tasks run in order and pass results along; `{placeholders}` and `inputs` make a crew reusable.
- A **Tool** is a Python function; its **docstring** is how the agent understands it.
- **Structured output** (`output_pydantic`) gives code a field it can use.
- A **Flow** gives you control: `@start` begins, `@listen` follows, `@router` branches, and `and_` / `or_` wait for several steps.
- Use **Crews to think and Flows to control.**

**Next:** open the longer lab, *Building AI Teams with CrewAI*, which applies these blocks to a small business with real data, then adds knowledge, memory, guardrails and a manager agent.